This Colab Notebook is the main script (refactored) of the master thesis 'Sources of Text Complexity for NMT'. In order to run it on your text:

- Click on the Files icon on the left & upload your text for analysis (.txt format)
- Replace input_file_name in cell 2 below with your file name
- Replace output-file_name with a name of your choice
- Click 'Run All' in the upper part of the Colab interface
- Wait
- Download json annotation of your text in the Files

In [20]:
# install dependencies
!pip install -q transformers huggingface_hub sympy

In [21]:
# rename variables (without file format)
input_file_name = "input_sample"
output_file_name = "output_sample"

In [22]:
# import necessary packages
import transformers
import numpy as np
import torch
import nltk
import json
from transformers import BertTokenizer, BertConfig
from transformers import BertForTokenClassification
from huggingface_hub import hf_hub_download
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [23]:
# define random seed to reproduce f1 score as reported in the study
torch.manual_seed(0)

# define tokenizer to tokenize your text for model processing
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case = True)


# download fine-tuned model from Hugging Face repo
model_path = hf_hub_download(
    repo_id="annaiankovskaia/bert-base-uncased-phrasal-verbs",
    filename="FINAL_BEST14.pt"
)

# use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the model
model = torch.load(model_path, map_location=device, weights_only = False)

In [24]:
# this part is refactoring for compatibility between the old fine-tuned model (2021) and new transformers in Colab (2026)

# remove generation parameters that belong to generation_config, not BertForTokenClassification config.
# newer transformers versions enforce this separation when saving

model.config.max_length = None
model.config.min_length = None
model.config.do_sample = None
model.config.early_stopping = None
model.config.num_beams = None
model.config.temperature = None
model.config.top_k = None
model.config.top_p = None
model.config.repetition_penalty = None
model.config.length_penalty = None
model.config.no_repeat_ngram_size = None
model.config.encoder_no_repeat_ngram_size = None
model.config.num_return_sequences = None
model.config.output_scores = None
model.config.return_dict_in_generate = None
model.config.remove_invalid_values = None
model.config.num_beam_groups = None
model.config.diversity_penalty = None

# fix missing internal config attributes caused by the current vs. 2021 Transformers versions mismatch

model.config._output_attentions = False
model.config._output_hidden_states = False
model.config._attn_implementation_internal = "eager"

# save the model in Hugging Face format
# this creates config.json + model weight shards, avoiding future torch/Transformers pickle compatibility issues

model.save_pretrained("fixed_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 MODULE 1: IDENTIFICATION OF PHRASAL VERBS. RUN THIS MODULE TO LABEL YOUR ENGLISH TEXT FOR PHRASAL VERBS

(with slight modifications, the code of Module 1 is taken from tutorial by Sterbak (2018) at https://www.depends-on-the-definition.com/named-entity-recognition-with-bert/

In [25]:
# mapping between annotation labels and numerical class IDs used by the model
# 'PV' means phrasal verb; '0' means no phrasal verb; 'PAD' - padding token (not written to the file but required for processing)

tag2idx = {'0': 1, 'PAD': 2, 'PV': 0}
tag_values = ['PV', '0', 'PAD']

In [26]:
# load the trained BERT token classification model and move it to the available
# computation device (GPU if available, otherwise CPU) for inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForTokenClassification.from_pretrained("fixed_model")
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [27]:
# initialize phrasal verb token counter
counter = 0

# read input file
file_path = f"/content/{input_file_name}.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

# split text into sentences for processing
sentences = sent_tokenize(text)

# create json structure
results = {
    "document": {
        "input_file": file_path.split("/")[-1],
        "total_sentences": len(sentences)
    },
    "sentences": [],
    "statistics": {
        "approximate_phrasal_verb_count": 0
    }
}

# process each sentence independently: tokenize it, run BERT inference,
# and convert model output probabilities into predicted label indices

for sentence_id, sentence in enumerate(sentences):
    tokenized_sentence = tokenizer.encode(sentence)
    input_ids = torch.tensor([tokenized_sentence]).to(device)
    with torch.no_grad():
        output = model(input_ids)
    label_indices = np.argmax(
        output[0].to('cpu').numpy(),
        axis=2
    )
    # convert ids back to tokens
    tokens = tokenizer.convert_ids_to_tokens(
        input_ids.to('cpu').numpy()[0]
    )
    # merge BPE split tokens
    new_tokens = []
    new_labels = []
    for token, label_idx in zip(tokens, label_indices[0]):
        label = tag_values[label_idx]

        # skip padding tokens
        if label == "PAD":
            continue
        if token.startswith("##"):
            new_tokens[-1] += token[2:]
        else:
            new_tokens.append(token)
            new_labels.append(label)

    # store sentence annotation
    sentence_data = {
        "id": sentence_id,
        "text": sentence,
        "tokens": []
    }

    # update json structure
    for token, label in zip(new_tokens, new_labels):
        sentence_data["tokens"].append({
            "token": token,
            "label": label
        })

        # update counter if phrasal verb
        if label == "PV":
            counter += 1
    results["sentences"].append(sentence_data)

# count and store total number of phrasal verbs (rule-based, thus approximate)
results["statistics"]["approximate_phrasal_verb_count"] = counter // 2

# save json
with open(f"{output_file_name}.json", "w", encoding="utf-8") as writefile:
    json.dump(
        results,
        writefile,
        indent=2,
        ensure_ascii=False
    )

MODULE 2: NOUN PHRASE DETECTION. RUN THIS MODULE TO DETECT IN YOUR TEXT NOUN PHRASES POTENTIALLY PROBLEMATIC FOR NMT FROM ENGLISH INTO RUSSIAN

Code for this module is based on the NLTK library (Bird, S., Loper, E. and Klein, E. (2009) Natural Language Processing with Python. O’Reilly
Media Inc.) and its significant part is taken from the Gist repository by Karimkhan (2016) at https://gist.github.com/karimkhanp/4b7626a933759d0113d54b09acef24bf

In [28]:
# define punctuation tokens that should be kept during POS tagging
exceptions = [',', '.', ';', ':', '-']

# extract noun phrase leaves from an NLTK chunk tree
def leaves(tree):
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
        yield subtree.leaves()

# extract noun phrase words and their POS tag patterns from the chunk tree
def get_np(tree):
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
        leaves = subtree.leaves()
        words = [w for w, t in leaves]
        pos_pattern = " ".join([t for w, t in leaves])
        yield words, pos_pattern

# define grammar rules for detecting noun phrase structures based on POS patterns
grammar = r"""
    NP:
        {<JJ.*>{1,}<NN.*>{1,}<IN><NN.*>{1,}}
        {<NN.*>{1,}<IN><JJ.*>{1,}<NN.*>{1,}}
        {<NN.*>{1,}<IN><NN.*>{2,}}
        {<NN.*>{2,}<IN><NN.*>{1,}}
        {<RB.*>{1,}<JJ.*>{1,}<NN.*>{1,}}
        {<JJ.*>*<NN.*>{2,}}
"""
# initialize counter for noun phrases and storage for complexity score calculation
counter2 = 0
list_for_score = []

# load json created by module 1
with open(f"{output_file_name}.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# create noun phrase chunk parser using the defined grammar rules
chunker = nltk.RegexpParser(grammar)

# process each sentence and add detected noun phrases to the existing json structure
for sentence_data in results["sentences"]:
    # tokenize sentence and store tokens for later complexity calculation
    sent = sentence_data["text"]
    toks = nltk.word_tokenize(sent)
    for tok in toks:
        list_for_score.append(tok)

    # assign POS tags and remove unwanted tokens before noun phrase parsing
    postoks = nltk.pos_tag(toks)
    postoks2 = []
    for postok in postoks:
        if (
            postok[0].isalnum()
            and len(postok[0]) > 1
        ) or any(el in postok[0] for el in exceptions):
            postoks2.append(postok)

    # parse POS-tagged tokens and extract noun phrase candidates
    tree = chunker.parse(postoks2)

    # initialize noun phrase list for the current sentence
    sentence_data["noun_phrases"] = []

    # store detected noun phrases and their POS patterns in JSON
    for np, pos_pattern in get_np(tree):
        phrase = " ".join(np)

        sentence_data["noun_phrases"].append({
            "text": phrase,
            "pos_pattern": pos_pattern
        })
        # update counter
        counter2 += 1

# add total noun phrase count to json statistics
results["statistics"]["noun_phrase_count"] = counter2

# save updated json with noun phrase annotations included
with open(f"{output_file_name}.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

# prepare cleaned token list for final complexity score calculation
tokens_list = [
    element
    for element in list_for_score
    if element.isalnum()
]

PART 3: CALCULATE PARTIAL TRANSLATION COMPLEXITY SCORE AND UPDATE JSON FILE

In [29]:
# define how score is calculated
def score(counter, counter2, tokens_list):
    counter = int(counter)
    counter2 = int(counter2)
    length = len(tokens_list)
    score = (counter + counter2) / length
    rounded = round(score, 2)
    return rounded

# calculate score
score_to_write = score(counter, counter2, tokens_list)

# load existing json
with open(f"{output_file_name}.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# add score
results["statistics"]["partial_complexity_score"] = score_to_write

# save updated json
with open(f"{output_file_name}.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )